# 29 prioritize liana axes

**Purpose:** Prioritize candidate communication axes and prepare final figures.

**Original analysis notebook:** `26_LIANA_final_prioritized_axes_and_figures_HVG8000_r0_6.ipynb`

Run this notebook from top to bottom in a fresh kernel. Project paths are
resolved from `GLIOMA_PROJECT_ROOT`; when that variable is not set, the
notebook searches parent directories for the repository root. Generated
AnnData objects are written under `data/processed/` or `data/interim/`,
while tables, figures, and logs retain the workflow's historical output
directories at the repository root.


# 45 — LIANA final prioritization and thesis-ready figures (HVG8000 r0.6)

این نوت‌بوک **LIANA را دوباره اجرا نمی‌کند**. هدف آن جمع‌بندی نهایی خروجی‌های LIANA global و condition-specific است تا محورهای ارتباطی قابل گزارش برای پایان‌نامه آماده شوند.

کارهای اصلی:

1. خواندن خروجی‌های LIANA global main با labelهای اصلاح‌شده.
2. خواندن خروجی‌های LIANA condition-specific.
3. ادغام و اولویت‌بندی محورهای زیستی مهم.
4. خلاصه‌سازی interactionهای candidate مثل `PTN/MDK–PTPRZ1/NCL`, `TIMP1–CD63`, `LGALS1–ITGB1`, `MIF–EGFR`, `VIM–CD44`, `VEGFA` و `APP–CD74`.
5. ساخت tableهای thesis-ready و figureهای نهایی.

نکته تفسیری مهم: این مرحله **differential communication آماری/replicate-aware** انجام نمی‌دهد. خروجی‌ها برای **prioritized candidate communication axes** هستند و باید همراه با annotation، CNV، state composition و program-score results تفسیر شوند.


In [ ]:

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 200)

import os

def _find_repository_root(start=None):
    """Locate the repository root or use GLIOMA_PROJECT_ROOT when set."""
    start = Path.cwd() if start is None else Path(start)
    start = start.expanduser().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "README.md").exists() and (candidate / "code").exists():
            return candidate
    return start

PROJECT_ROOT = Path(
    os.environ.get("GLIOMA_PROJECT_ROOT", _find_repository_root())
).expanduser().resolve()
METADATA_DIR = PROJECT_ROOT / 'metadata'
FIGURE_DIR = PROJECT_ROOT / 'figures' / 'LIANA_final_prioritization_HVG8000_r0_6'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

OUT_PREFIX = 'LIANA_final_prioritization_HVG8000_r0_6'

GLOBAL_PREFIX = 'LIANA_global_main_HVG8000_r0_6_clean_labels'
COND_PREFIX = 'LIANA_condition_specific_HVG8000_r0_6'

print('PROJECT_ROOT:', PROJECT_ROOT)
print('METADATA_DIR:', METADATA_DIR)
print('FIGURE_DIR:', FIGURE_DIR)


In [ ]:

def read_tsv(path, required=False):
    path = Path(path)
    if not path.exists():
        msg = f'[MISSING] {path}'
        if required:
            raise FileNotFoundError(msg)
        print(msg)
        return pd.DataFrame()
    df = pd.read_csv(path, sep='\t')
    print(f'[READ] {path.name}: {df.shape}')
    return df


def write_tsv(df, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, sep='\t', index=False)
    print(f'[SAVED] {path} shape={df.shape}')
    return path


def save_fig(path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    plt.tight_layout()
    plt.savefig(path, dpi=220, bbox_inches='tight')
    plt.close()
    print(f'[SAVED FIG] {path}')
    return path


def safe_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None


def rank_col(df):
    for c in ['rank_for_sorting', 'primary_rank', 'magnitude_rank', 'specificity_rank', 'median_rank', 'median_primary_rank', 'min_rank', 'min_primary_rank']:
        if c in df.columns:
            return c
    return None


def clean_lr_pair(row):
    if 'lr_pair' in row and pd.notna(row['lr_pair']):
        return str(row['lr_pair'])
    lig = str(row.get('ligand', row.get('ligand_complex', '')))
    rec = str(row.get('receptor', row.get('receptor_complex', '')))
    return f'{lig}→{rec}'


def simplify_lr_pair(lr):
    lr = str(lr)
    lr = lr.replace('_', '-') if lr.count('→') == 0 else lr
    return lr


def axis_display_name(axis):
    if pd.isna(axis):
        return 'unknown'
    axis = str(axis)
    mapping = {
        'malignant_or_cycling_to_malignant_or_cycling': 'Malignant/cycling → malignant/cycling',
        'malignant_or_cycling_to_microenvironment': 'Malignant/cycling → microenvironment',
        'microenvironment_to_malignant_or_cycling': 'Microenvironment → malignant/cycling',
        'microenvironment_to_microenvironment': 'Microenvironment → microenvironment',
        'malignant_to_microenvironment': 'Malignant/cycling → microenvironment',
        'microenvironment_to_malignant': 'Microenvironment → malignant/cycling',
    }
    return mapping.get(axis, axis.replace('_', ' → '))


## 1. Read LIANA global and condition-specific summary files

In [ ]:

# Global clean-label outputs
global_cell_counts = read_tsv(METADATA_DIR / f'{GLOBAL_PREFIX}_celltype_counts_used.tsv', required=True)
global_source_target = read_tsv(METADATA_DIR / f'{GLOBAL_PREFIX}_source_target_summary.tsv', required=True)
global_axis = read_tsv(METADATA_DIR / f'{GLOBAL_PREFIX}_axis_summary.tsv', required=True)
global_bio_axis = read_tsv(METADATA_DIR / f'{GLOBAL_PREFIX}_biological_axis_summary.tsv', required=True)
global_lr_summary = read_tsv(METADATA_DIR / f'{GLOBAL_PREFIX}_ligand_receptor_summary.tsv', required=True)
global_focused_axis = read_tsv(METADATA_DIR / f'{GLOBAL_PREFIX}_focused_axis_summary.tsv', required=False)

# Condition-specific outputs
cond_run = read_tsv(METADATA_DIR / f'{COND_PREFIX}_run_summary.tsv', required=True)
cond_axis = read_tsv(METADATA_DIR / f'{COND_PREFIX}_axis_summary.tsv', required=True)
cond_bio_axis = read_tsv(METADATA_DIR / f'{COND_PREFIX}_biological_axis_summary.tsv', required=True)
cond_lr_summary = read_tsv(METADATA_DIR / f'{COND_PREFIX}_ligand_receptor_summary.tsv', required=True)
cond_source_target = read_tsv(METADATA_DIR / f'{COND_PREFIX}_source_target_summary.tsv', required=True)
cond_presence = read_tsv(METADATA_DIR / f'{COND_PREFIX}_prioritized_interaction_condition_presence.tsv', required=True)
cond_prioritized = read_tsv(METADATA_DIR / f'{COND_PREFIX}_prioritized_interactions.tsv', required=True)
cond_candidate_axis = read_tsv(METADATA_DIR / f'{COND_PREFIX}_candidate_axis_interaction_summary.tsv', required=False)

readiness = pd.DataFrame([
    {'item': 'global_cell_counts_rows', 'value': len(global_cell_counts)},
    {'item': 'global_prioritized_lr_pairs', 'value': len(global_lr_summary)},
    {'item': 'condition_specific_runs', 'value': len(cond_run)},
    {'item': 'condition_specific_prioritized_rows', 'value': len(cond_prioritized)},
    {'item': 'condition_presence_rows', 'value': len(cond_presence)},
])
write_tsv(readiness, METADATA_DIR / f'{OUT_PREFIX}_readiness.tsv')

display(readiness)
display(cond_run)


## 2. Define candidate ligand–receptor axes for thesis-focused prioritization

In [ ]:

# Candidate LR axes selected from global and condition-specific LIANA summaries.
# These are not claimed as functionally proven interactions; they are prioritized candidates.

CANDIDATE_LR_PATTERNS = {
    'PTN_PTPRZ1_NCL_SDC3_axis': [
        ('PTN', 'PTPRZ1'), ('PTN', 'NCL'), ('PTN', 'SDC3'), ('PTN', 'ALK'), ('PTN', 'SDC4')
    ],
    'MDK_PTPRZ1_NCL_axis': [
        ('MDK', 'PTPRZ1'), ('MDK', 'NCL'), ('MDK', 'SDC3'), ('MDK', 'SDC4'), ('MDK', 'LRP1')
    ],
    'TIMP1_CD63_axis': [('TIMP1', 'CD63')],
    'LGALS1_ITGB1_axis': [('LGALS1', 'ITGB1')],
    'MIF_EGFR_CD74_axis': [('MIF', 'EGFR'), ('MIF', 'CD74'), ('MIF', 'CXCR4'), ('MIF', 'CD44')],
    'VIM_CD44_axis': [('VIM', 'CD44')],
    'VEGFA_angiogenesis_axis': [('VEGFA', 'FLT1'), ('VEGFA', 'KDR'), ('VEGFA', 'NRP1'), ('VEGFA', 'ITGB1')],
    'APP_CD74_axis': [('APP', 'CD74')],
    'HLA_A_APLP2_cautious_axis': [('HLA-A', 'APLP2')],
}

CAUTION_AXES = {'HLA_A_APLP2_cautious_axis'}


def infer_candidate_axis(ligand, receptor, lr_pair=None):
    ligand = str(ligand) if pd.notna(ligand) else ''
    receptor = str(receptor) if pd.notna(receptor) else ''
    lr_pair = str(lr_pair) if lr_pair is not None and pd.notna(lr_pair) else f'{ligand}→{receptor}'
    hits = []
    for axis, pairs in CANDIDATE_LR_PATTERNS.items():
        for lig, rec in pairs:
            if ligand == lig and receptor == rec:
                hits.append(axis)
            elif lr_pair == f'{lig}→{rec}':
                hits.append(axis)
    if hits:
        return ';'.join(sorted(set(hits)))
    return ''


def is_candidate_row(row):
    axis = infer_candidate_axis(
        row.get('ligand', row.get('ligand_complex', '')),
        row.get('receptor', row.get('receptor_complex', '')),
        row.get('lr_pair', None)
    )
    return axis

candidate_inventory = []
for axis, pairs in CANDIDATE_LR_PATTERNS.items():
    for lig, rec in pairs:
        candidate_inventory.append({
            'candidate_axis': axis,
            'ligand': lig,
            'receptor': rec,
            'lr_pair': f'{lig}→{rec}',
            'interpretation_caution': 'high' if axis in CAUTION_AXES else 'standard'
        })
candidate_inventory = pd.DataFrame(candidate_inventory)
write_tsv(candidate_inventory, METADATA_DIR / f'{OUT_PREFIX}_candidate_lr_axis_inventory.tsv')
display(candidate_inventory)


## 3. Global LIANA summary for final reporting

In [ ]:

# Rank global biological axes for thesis reporting.

gbio = global_bio_axis.copy()
if not gbio.empty:
    if 'n_interactions' not in gbio.columns and 'n_prioritized_interactions' in gbio.columns:
        gbio = gbio.rename(columns={'n_prioritized_interactions': 'n_interactions'})
    gbio['axis_display'] = gbio['biological_axis'].astype(str)
    sort_cols = [c for c in ['n_interactions', 'n_source_target_pairs'] if c in gbio.columns]
    if sort_cols:
        gbio = gbio.sort_values(sort_cols, ascending=[False] * len(sort_cols))

global_top_axes = gbio.head(30).copy()
write_tsv(global_top_axes, METADATA_DIR / f'{OUT_PREFIX}_global_top_biological_axes.tsv')

gaxis = global_axis.copy()
if not gaxis.empty:
    gaxis['axis_display'] = gaxis['communication_axis'].map(axis_display_name)
write_tsv(gaxis, METADATA_DIR / f'{OUT_PREFIX}_global_communication_axis_summary.tsv')

# Global candidate LR pairs.
glr = global_lr_summary.copy()
if not glr.empty:
    if 'lr_pair' not in glr.columns:
        glr['lr_pair'] = glr.apply(clean_lr_pair, axis=1)
    glr['candidate_axis'] = glr.apply(is_candidate_row, axis=1)
    glr_candidates = glr[glr['candidate_axis'].astype(str).str.len() > 0].copy()
else:
    glr_candidates = pd.DataFrame()

write_tsv(glr_candidates, METADATA_DIR / f'{OUT_PREFIX}_global_candidate_lr_summary.tsv')

display(global_top_axes.head(15))
display(glr_candidates.head(30))


## 4. Condition-specific comparison summaries

In [ ]:

# Condition axis comparison table.

caxis = cond_axis.copy()
if not caxis.empty:
    caxis['axis_display'] = caxis['axis'].map(axis_display_name)
    axis_pivot = caxis.pivot_table(
        index='axis_display',
        columns='condition',
        values='n_interactions',
        aggfunc='sum',
        fill_value=0
    ).reset_index()
    axis_pivot['max_interactions'] = axis_pivot.drop(columns=['axis_display']).max(axis=1)
    axis_pivot = axis_pivot.sort_values('max_interactions', ascending=False)
else:
    axis_pivot = pd.DataFrame()

write_tsv(caxis, METADATA_DIR / f'{OUT_PREFIX}_condition_axis_long.tsv')
write_tsv(axis_pivot, METADATA_DIR / f'{OUT_PREFIX}_condition_axis_comparison.tsv')

# Biological axis condition comparison.
cbio = cond_bio_axis.copy()
if not cbio.empty:
    # keep top condition-specific biological axes for thesis. Top by max interactions across conditions.
    bio_pivot = cbio.pivot_table(
        index='biological_axis',
        columns='condition',
        values='n_interactions',
        aggfunc='sum',
        fill_value=0
    ).reset_index()
    condition_cols = [c for c in bio_pivot.columns if c != 'biological_axis']
    bio_pivot['max_interactions'] = bio_pivot[condition_cols].max(axis=1) if condition_cols else 0
    bio_pivot['total_interactions'] = bio_pivot[condition_cols].sum(axis=1) if condition_cols else 0
    bio_pivot = bio_pivot.sort_values(['max_interactions', 'total_interactions'], ascending=False)
else:
    bio_pivot = pd.DataFrame()

write_tsv(bio_pivot, METADATA_DIR / f'{OUT_PREFIX}_condition_biological_axis_comparison.tsv')

display(axis_pivot.head(20))
display(bio_pivot.head(30))


## 5. Candidate LR condition presence and source-target details

In [ ]:

# Candidate interaction presence across conditions.

pres = cond_presence.copy()
if not pres.empty:
    if 'lr_pair' not in pres.columns:
        pres['lr_pair'] = pres.apply(clean_lr_pair, axis=1)
    pres['candidate_axis'] = pres.apply(is_candidate_row, axis=1)
    pres_candidates = pres[pres['candidate_axis'].astype(str).str.len() > 0].copy()
    # Add caution flag
    pres_candidates['interpretation_caution'] = pres_candidates['candidate_axis'].apply(
        lambda x: 'high' if any(ax in CAUTION_AXES for ax in str(x).split(';')) else 'standard'
    )
    sort_cols = [c for c in ['n_conditions', 'min_rank_all', 'median_rank_all'] if c in pres_candidates.columns]
    if sort_cols:
        asc = [False if c == 'n_conditions' else True for c in sort_cols]
        pres_candidates = pres_candidates.sort_values(sort_cols, ascending=asc)
else:
    pres_candidates = pd.DataFrame()

write_tsv(pres_candidates, METADATA_DIR / f'{OUT_PREFIX}_candidate_lr_condition_presence.tsv')

# Candidate rows from condition-specific prioritized interactions.
cp = cond_prioritized.copy()
if not cp.empty:
    if 'lr_pair' not in cp.columns:
        cp['lr_pair'] = cp.apply(clean_lr_pair, axis=1)
    cp['candidate_axis'] = cp.apply(is_candidate_row, axis=1)
    cp_candidates = cp[cp['candidate_axis'].astype(str).str.len() > 0].copy()
    # Keep a compact table for upload/reading.
    keep_cols = [c for c in [
        'condition', 'candidate_axis', 'ligand', 'receptor', 'lr_pair',
        'source_clean', 'target_clean', 'source_target', 'axis', 'biological_axis',
        'rank_for_sorting', 'magnitude_rank', 'specificity_rank', 'lr_means', 'expr_prod',
        'scaled_weight', 'lr_logfc', 'prioritization_rule'
    ] if c in cp_candidates.columns]
    cp_candidates_compact = cp_candidates[keep_cols].copy()
    sort_cols = [c for c in ['candidate_axis', 'condition', 'rank_for_sorting'] if c in cp_candidates_compact.columns]
    if sort_cols:
        cp_candidates_compact = cp_candidates_compact.sort_values(sort_cols)
else:
    cp_candidates_compact = pd.DataFrame()

write_tsv(cp_candidates_compact, METADATA_DIR / f'{OUT_PREFIX}_candidate_lr_source_target_details.tsv')

display(pres_candidates.head(40))
display(cp_candidates_compact.head(40))


## 6. Build thesis-ready interpretation table

In [ ]:

# Create a compact interpretive table combining biological axis, recurrence, and caution.

interpretation_rows = []

# Use candidate presence to build concise summaries.
if not pres_candidates.empty:
    for axis, sub in pres_candidates.groupby('candidate_axis'):
        lr_pairs = '; '.join(sorted(sub['lr_pair'].dropna().astype(str).unique())[:12])
        n_keys = len(sub)
        conditions = sorted(set(';'.join(sub.get('conditions', pd.Series(dtype=str)).dropna().astype(str)).replace(',', ';').split(';')))
        conditions = [c for c in conditions if c]
        n_cond_max = sub['n_conditions'].max() if 'n_conditions' in sub.columns else np.nan
        top_bio = '; '.join(sub.get('biological_axis', pd.Series(dtype=str)).dropna().astype(str).value_counts().head(5).index.tolist())
        interpretation_rows.append({
            'candidate_axis': axis,
            'n_candidate_interaction_keys': n_keys,
            'max_n_conditions_detected': n_cond_max,
            'conditions_observed': ';'.join(conditions),
            'representative_lr_pairs': lr_pairs,
            'dominant_biological_axes': top_bio,
            'interpretation_caution': 'high' if axis in CAUTION_AXES else 'standard',
            'suggested_thesis_interpretation': {
                'PTN_PTPRZ1_NCL_SDC3_axis': 'Candidate malignant-state communication axis linked to glioma-associated growth/progenitor signaling; prioritize as recurrent malignant/microenvironment communication candidate.',
                'MDK_PTPRZ1_NCL_axis': 'Candidate growth-factor-like communication axis, often related to malignant and stromal communication; interpret as putative signaling candidate.',
                'TIMP1_CD63_axis': 'Candidate ECM/remodeling and stress-associated communication axis, prominent across multiple sender-target contexts.',
                'LGALS1_ITGB1_axis': 'Candidate adhesion/ECM-associated communication axis connecting malignant and microenvironmental compartments.',
                'MIF_EGFR_CD74_axis': 'Candidate inflammatory/growth signaling axis; interpret with caution because MIF can act through multiple receptor complexes.',
                'VIM_CD44_axis': 'Candidate mesenchymal/stress-associated communication axis, consistent with reactive/MES-like programs.',
                'VEGFA_angiogenesis_axis': 'Candidate angiogenic communication axis, especially relevant when involving endothelial or mural/pericyte-like targets.',
                'APP_CD74_axis': 'Candidate immune/inflammatory communication axis; interpret as prioritized LR signal rather than direct functional proof.',
                'HLA_A_APLP2_cautious_axis': 'Cautious candidate; HLA-related interactions may reflect broad expression/antigen-presentation signals and should not be overinterpreted.'
            }.get(axis, 'Candidate LIANA-prioritized ligand–receptor axis.')
        })

interpretation = pd.DataFrame(interpretation_rows)
if not interpretation.empty:
    interpretation = interpretation.sort_values(['max_n_conditions_detected', 'n_candidate_interaction_keys'], ascending=False)

write_tsv(interpretation, METADATA_DIR / f'{OUT_PREFIX}_thesis_candidate_axis_interpretation.tsv')
display(interpretation)


## 7. Figures: global axes, condition axes, and candidate LR presence

In [ ]:

figure_records = []

# Figure 1: Global biological axes barplot
if not global_top_axes.empty and 'n_interactions' in global_top_axes.columns:
    dfp = global_top_axes.head(15).copy()
    labels = dfp['biological_axis'].astype(str).str.replace('_', ' ', regex=False)
    plt.figure(figsize=(9, max(4, 0.35 * len(dfp))))
    plt.barh(range(len(dfp)), dfp['n_interactions'])
    plt.yticks(range(len(dfp)), labels)
    plt.gca().invert_yaxis()
    plt.xlabel('Number of prioritized LIANA interactions')
    plt.title('Global LIANA prioritized biological axes')
    p = save_fig(FIGURE_DIR / f'{OUT_PREFIX}_global_biological_axes_barplot.png')
    figure_records.append({'figure': 'global_biological_axes_barplot', 'path': str(p)})

# Figure 2: Condition communication axis heatmap
if not axis_pivot.empty:
    dfh = axis_pivot.copy()
    condition_cols = [c for c in ['GBM', 'LGG', 'ndGBM', 'rGBM'] if c in dfh.columns]
    if condition_cols:
        plot_df = dfh.set_index('axis_display')[condition_cols]
        plt.figure(figsize=(8, max(3, 0.55 * len(plot_df))))
        plt.imshow(plot_df.values, aspect='auto')
        plt.xticks(range(len(condition_cols)), condition_cols, rotation=45, ha='right')
        plt.yticks(range(len(plot_df.index)), plot_df.index)
        plt.colorbar(label='Number of prioritized interactions')
        plt.title('Condition-specific LIANA communication axes')
        p = save_fig(FIGURE_DIR / f'{OUT_PREFIX}_condition_axis_heatmap.png')
        figure_records.append({'figure': 'condition_axis_heatmap', 'path': str(p)})

# Figure 3: Candidate LR condition presence heatmap by min rank converted to -log10(rank)
if not pres_candidates.empty:
    pc = pres_candidates.copy()
    # Use one row per interaction key. Prefer rank columns per condition if available.
    rank_cols = [c for c in ['rank_GBM', 'rank_LGG', 'rank_ndGBM', 'rank_rGBM'] if c in pc.columns]
    if rank_cols:
        pc['display_key'] = pc['lr_pair'].astype(str) + ' | ' + pc.get('source_clean', '').astype(str).str.replace('Malignant_', 'M_', regex=False).str.replace('NonMalignant_', 'NM_', regex=False) + '→' + pc.get('target_clean', '').astype(str).str.replace('Malignant_', 'M_', regex=False).str.replace('NonMalignant_', 'NM_', regex=False)
        pc = pc.sort_values(['n_conditions', 'min_rank_all'], ascending=[False, True]).head(35)
        mat = pc[rank_cols].astype(float).replace([np.inf, -np.inf], np.nan)
        score = -np.log10(mat.clip(lower=1e-12))
        score = score.fillna(0)
        labels = [c.replace('rank_', '') for c in rank_cols]
        plt.figure(figsize=(9, max(5, 0.25 * len(pc))))
        plt.imshow(score.values, aspect='auto')
        plt.xticks(range(len(labels)), labels, rotation=45, ha='right')
        plt.yticks(range(len(pc)), pc['display_key'])
        plt.colorbar(label='-log10(rank); 0 = not prioritized/detected')
        plt.title('Candidate LIANA interactions across conditions')
        p = save_fig(FIGURE_DIR / f'{OUT_PREFIX}_candidate_lr_condition_presence_heatmap.png')
        figure_records.append({'figure': 'candidate_lr_condition_presence_heatmap', 'path': str(p)})

# Figure 4: Candidate axis counts
if not interpretation.empty:
    dfp = interpretation.copy()
    plt.figure(figsize=(9, max(4, 0.45 * len(dfp))))
    plt.barh(range(len(dfp)), dfp['n_candidate_interaction_keys'])
    plt.yticks(range(len(dfp)), dfp['candidate_axis'])
    plt.gca().invert_yaxis()
    plt.xlabel('Number of condition-specific candidate interaction keys')
    plt.title('Final prioritized LIANA candidate axes')
    p = save_fig(FIGURE_DIR / f'{OUT_PREFIX}_candidate_axis_counts_barplot.png')
    figure_records.append({'figure': 'candidate_axis_counts_barplot', 'path': str(p)})

figure_manifest = pd.DataFrame(figure_records)
write_tsv(figure_manifest, METADATA_DIR / f'{OUT_PREFIX}_figure_manifest.tsv')
display(figure_manifest)


## 8. Final summary and output manifest

In [ ]:

# Final compact run summary

summary_rows = []
summary_rows.append({'item': 'global_celltypes_used', 'value': global_cell_counts.shape[0]})
summary_rows.append({'item': 'global_top_biological_axes_saved', 'value': len(global_top_axes)})
summary_rows.append({'item': 'condition_runs_completed', 'value': int((cond_run.get('status', pd.Series(dtype=str)) == 'completed').sum()) if not cond_run.empty else 0})
summary_rows.append({'item': 'condition_prioritized_rows', 'value': len(cond_prioritized)})
summary_rows.append({'item': 'candidate_lr_condition_presence_rows', 'value': len(pres_candidates)})
summary_rows.append({'item': 'candidate_lr_source_target_detail_rows', 'value': len(cp_candidates_compact)})
summary_rows.append({'item': 'candidate_axes_prioritized', 'value': interpretation['candidate_axis'].nunique() if not interpretation.empty else 0})
summary_rows.append({'item': 'figures_created', 'value': len(figure_manifest)})
summary_rows.append({'item': 'interpretation_note', 'value': 'Candidate communication axes only; not functional proof and not replicate-aware differential communication.'})
summary = pd.DataFrame(summary_rows)
write_tsv(summary, METADATA_DIR / f'{OUT_PREFIX}_run_summary.tsv')

# Final output manifest
expected_outputs = [
    METADATA_DIR / f'{OUT_PREFIX}_readiness.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_candidate_lr_axis_inventory.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_global_top_biological_axes.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_global_communication_axis_summary.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_global_candidate_lr_summary.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_condition_axis_long.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_condition_axis_comparison.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_condition_biological_axis_comparison.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_candidate_lr_condition_presence.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_candidate_lr_source_target_details.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_thesis_candidate_axis_interpretation.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_figure_manifest.tsv',
    METADATA_DIR / f'{OUT_PREFIX}_run_summary.tsv',
]
manifest = []
for p in expected_outputs:
    p = Path(p)
    manifest.append({'path': str(p), 'exists': p.exists(), 'size_mb': round(p.stat().st_size / 1024**2, 3) if p.exists() else np.nan})
for p in figure_manifest['path'].tolist() if not figure_manifest.empty else []:
    p = Path(p)
    manifest.append({'path': str(p), 'exists': p.exists(), 'size_mb': round(p.stat().st_size / 1024**2, 3) if p.exists() else np.nan})
manifest = pd.DataFrame(manifest)
write_tsv(manifest, METADATA_DIR / f'{OUT_PREFIX}_output_manifest.tsv')

display(summary)
display(manifest)



## Outputs to send back

بعد از اجرای کامل این نوت‌بوک، این فایل‌ها را بفرست:

```text
${GLIOMA_PROJECT_ROOT}/metadata/LIANA_final_prioritization_HVG8000_r0_6_readiness.tsv
${GLIOMA_PROJECT_ROOT}/metadata/LIANA_final_prioritization_HVG8000_r0_6_global_top_biological_axes.tsv
${GLIOMA_PROJECT_ROOT}/metadata/LIANA_final_prioritization_HVG8000_r0_6_condition_axis_comparison.tsv
${GLIOMA_PROJECT_ROOT}/metadata/LIANA_final_prioritization_HVG8000_r0_6_condition_biological_axis_comparison.tsv
${GLIOMA_PROJECT_ROOT}/metadata/LIANA_final_prioritization_HVG8000_r0_6_candidate_lr_condition_presence.tsv
${GLIOMA_PROJECT_ROOT}/metadata/LIANA_final_prioritization_HVG8000_r0_6_candidate_lr_source_target_details.tsv
${GLIOMA_PROJECT_ROOT}/metadata/LIANA_final_prioritization_HVG8000_r0_6_thesis_candidate_axis_interpretation.tsv
${GLIOMA_PROJECT_ROOT}/metadata/LIANA_final_prioritization_HVG8000_r0_6_run_summary.tsv
${GLIOMA_PROJECT_ROOT}/metadata/LIANA_final_prioritization_HVG8000_r0_6_figure_manifest.tsv
${GLIOMA_PROJECT_ROOT}/metadata/LIANA_final_prioritization_HVG8000_r0_6_output_manifest.tsv
```

اگر figureها خوب ساخته شدند، اسکرین‌شات heatmapهای candidate LR و condition axis را هم بفرست.
